# CNN in PyTorch

**Run locally** — this notebook requires PyTorch and torchvision.

Implements the Atari DQN CNN architecture and trains on MNIST for verification.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt


## 1. Atari DQN-style CNN architecture

In [ ]:
class AtariCNN(nn.Module):
    """Simplified Atari DQN CNN. Original uses 84x84 input; we use 28x28."""
    def __init__(self, n_actions=10):
        super().__init__()
        # Conv layers (like DQN but smaller for MNIST)
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, stride=2)  # 28x28 -> 12x12
        self.conv2 = nn.Conv2d(16, 32, kernel_size=4, stride=1)  # 12x12 -> 9x9
        self.conv3 = nn.Conv2d(32, 32, kernel_size=3, stride=1)  # 9x9 -> 7x7
        # FC layers
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, n_actions)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)  # flatten
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = AtariCNN(n_actions=10)
print(model)
print('\nParameters:', sum(p.numel() for p in model.parameters()))

# Test with dummy MNIST-sized input
dummy = torch.randn(4, 1, 28, 28)  # batch=4, channels=1, 28x28
out = model(dummy)
print('Output shape:', out.shape, '  (4 samples, 10 class logits)')


## 2. Train on MNIST

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1000)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AtariCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(epoch):
    model.train()
    total_loss = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def test():
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            correct += output.argmax(dim=1).eq(target).sum().item()
    return correct / len(test_data)

losses = []
for epoch in range(3):
    loss = train_epoch(epoch)
    acc = test()
    losses.append(loss)
    print(f'Epoch {epoch+1}: loss={loss:.4f}, test accuracy={acc*100:.1f}%')


## 3. Visualize learned filters

In [ ]:
filters = model.conv1.weight.data.cpu()
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    if i < filters.shape[0]:
        ax.imshow(filters[i, 0], cmap='RdBu')
    ax.axis('off')
plt.suptitle('Conv1 learned filters (16 filters, 5x5)')
plt.tight_layout(); plt.show()
